# ===============================
#  Define the All Prompt
# ===============================

# Define a Prompt to generate Question
from langchain_core.prompts import PromptTemplate

# Prompt template (Key Note: Specific your tone what are exactly you want to be Create)
prompt_template = """
You are an expert at creating questions based on coding materials and documentation.
Your goal is to prepare a coder or programmer for their exam and coding tests.
You do this by asking questions about the text below:

---------
{text}
---------

Create questions that will prepared the coders or programmers for their test.
Make sure not to lose any important information.

QUESTIONS:

"""


# Prompt template 2 (here, inputs: existing question, chunk of documents)
refine_template = ("""
You are an expert at creating practice questions based on coding material and documentation.

Generate interview questions based strictly on the provided context.

Question requirements:
- Questions must be contextual and derived from the provided text.
- Prefer scenario-based, application-oriented, and reasoning-based questions.
- Avoid simple definition-recall questions.
- Avoid questions that can be answered without understanding the provided context.
- Do not invent information outside the provided context.
- Do not include explanations, headings, answers, or refinement notes.

Generate:
- 5 conceptual/contextual questions
- 3 scenario-based questions
- 2 reasoning/application questions

Return ONLY the questions as a numbered list.

Your goal is to help a coder or programmer prepare for a coding test.
We have received some practice questions to a certain extent: {existing_answer}.
We have the option to refine the existing questions or add new ones.
(only if necessary) with some more context below.
------------
{text}
------------

Given the new context on Creative Questions or Contextual Questions, refine the original questions in English.
If the context is not helpful, please provide the original questions.
QUESTIONS:


"""
)



# Define the prompt that handles the context documents
system_prompt = (
""" 
You are a technical assistant. Give the shortest correct answer that fully addresses the question. Use bullet points only when listing multiple distinct items; otherwise use plain sentences. No introductory phrases like "Great question!" — start directly with the answer.

Answer the user's question using only the provided context. Respond in 1-3 sentences. If the context doesn't contain the answer, say so in one sentence — do not guess or use outside knowledge. Do not repeat the question back.
Context:\n{context}    

"""
)



In [ ]:

# =========================================================
# STEP 1: Define All Require Libraries
# =========================================================
import os
from dotenv import load_dotenv
from src.prompt import *
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough



# =========================================================
# STEP 2: Groq authentication
# =========================================================
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# =========================================================
# STEP 3: Load PDF Document, Chunk Distribution
# =========================================================

def file_processing(file_path):
    """ 
    Method name: file_processing
    input : file path
    output: 
        - document_ques_gen: Return all document of 1st layer

        - document_answer_gen: Return Chunk of document of 2nd layer
    """

    # ---------------------------------------------------------
    # Step 3.1: Load data from PDF
    # ---------------------------------------------------------
    loader = PyPDFLoader(file_path)
    data = loader.load()

    # # formatting the data
    question_gen = ''
    for page in data:
        question_gen += page.page_content

    # ---------------------------------------------------------
    # Step 3.2: 1st Layer of Chunk Distribution
    # ---------------------------------------------------------
    splitter_ques_gen = RecursiveCharacterTextSplitter(
        # model_name = "openai/gpt-oss-20b",
        chunk_size=10000,
        chunk_overlap=200
    )

    chunks_ques_gen = splitter_ques_gen.split_text(question_gen)


    # ---------------------------------------------------------
    # Step 3.3: 2nd Layer of Chunk Distribution
    # ---------------------------------------------------------
    document_ques_gen = [Document(page_content=t) for t in chunks_ques_gen]
    
    splitter_ans_gen = RecursiveCharacterTextSplitter(
        # model_name = "openai/gpt-oss-20b",
        chunk_size=1000,
        chunk_overlap=200
    )

    document_answer_gen = splitter_ans_gen.split_documents(
        document_ques_gen
    )

    return document_ques_gen, document_answer_gen

# =========================================================
# STEP 4: llm pipeline 
# 
# =========================================================
def llm_pipeline(file_path):
    """ 
    input: file path
    output: 
        - return the all Questions answer: answer_generation_chain
        - Return the all Questions list: all_questions_lst
        
    """
    # ---------------------------------------------------------------------
    # Step 4.0: processing the provide file using file_processing function
    # ---------------------------------------------------------------------

    document_ques_gen, document_answer_gen = file_processing(file_path)

    # ---------------------------------------------------------
    # Step 4.1: Define the LLM to generate question
    # ---------------------------------------------------------
    llm_ques_gen_pipeline = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0.3
    )

    # -----------------------------------------------------------------------
    # Step 4.2: Define the Prompt 1 -> to create generate Questions
    # -----------------------------------------------------------------------
    PROMPT_QUESTIONS = PromptTemplate(template=prompt_template, input_variables=["text"])

    
    # -----------------------------------------------------------------------
    # Step 4.3: Define the Prompt 2 -> to create Create Refine Questions
    # -----------------------------------------------------------------------
    REFINE_PROMPT_QUESTIONS = PromptTemplate(
        input_variables=["existing_answer", "text"],
        template=refine_template,
    )

    # -----------------------------------------------------------------------
    # Step 4.4: Define the Chian  -> to create Create Refine Questions
    # -----------------------------------------------------------------------
    Generic_Question_chain = (
        PROMPT_QUESTIONS
        | llm_ques_gen_pipeline
        | StrOutputParser()
    )

    ques_gen_chain = (
        {
            "existing_answer": Generic_Question_chain,
            "text": RunnablePassthrough()
        }
        | REFINE_PROMPT_QUESTIONS
        | StrOutputParser()

    )
    ques = ques_gen_chain.invoke(document_ques_gen)

    # -----------------------------------------------------------------------
    # Step 4.5: Define the Embedding Model
    # -----------------------------------------------------------------------
    embeddings =  HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    # -----------------------------------------------------------------------
    # Step 4.5: Define the Vector Store (Knowledge Base)
    # -----------------------------------------------------------------------
    vector_store = FAISS.from_documents(document_answer_gen, embeddings)

    # ---------------------------------------------------------
    # Step 4.6: Define the LLM for Answer
    # ---------------------------------------------------------
    llm_answer_gen = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0.1
    )

    # Store the generated questions from all document chunks
    all_questions_lst = []
    for document in ques:
        text = document.page_content
        questions = llm_answer_gen.invoke(text)
        all_questions_lst.append(questions)

    # ---------------------------------------------------------
    # Step 4.7: Create a Retrieval
    # ---------------------------------------------------------
    retriever = vector_store.as_retriever(
        search_type = "similarity",
        search_kwargs = {
        "k": 4,
        }
    )

    # Helper to format retrieved documents into a single text block
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)


    # --------------------------------------------------------------------
    # Step 4.8: Define the Prompt for Generate Answer of the questions
    # --------------------------------------------------------------------
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
    ])


    # ---------------------------------------------------------
    # Step 4.9: Create a Chain for Answer the questions
    # ---------------------------------------------------------
    answer_generation_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt
        | llm_answer_gen
        | StrOutputParser()
    )

    return answer_generation_chain, all_questions_lst

In [ ]:
## This is templates/index.html file
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Document</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.1/dist/css/bootstrap.min.css" rel="stylesheet" integrity="sha384-4bw+/aepP/YC94hEpVNVgiZdgIC5+VKNBQNGCHeKRQN+PtmoHDEXuppvnDJzQIu9" crossorigin="anonymous">
    
    <style>
        #result, #download {
            display: none;
        }

        .font-large {
            font-size: 150px;
        }
    </style>
</head>
<body class="bg-dark">
    <section>
        <div class="container-fluid">
            <div class="row">
                <div class="cl-sm-12 text-center p-5 text-white">
                    <h3>Interview Question Creator</h3>
                </div>
            </div>

        </div>
    </section>
    <section class="mb-4">
        <div class="container">
            <div class="row">
                <div class="cl-sm-12">
                    <div class="card p-5 shadow border-0 m-3">
                        <div class="mb-3">
                            <label for="exampleFormControlInput1" class="form-label">Upload your PDF file here</label>
                            <div class="input-group mb-3">
                                <input type="file" class="form-control" id="pdf-file">
                                <label class="input-group-text" for="pdf-file">Max No. of Pages is 5</label>
                            </div>
                          </div>
                          <div class="mb-3 text-end">
                            <button type="button" id="upload-btn" class="btn btn-md btn-success">Generate Q&A</button>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </section>
    <section id="result">
        <div class="container">
            <div class="row">
                <div class="col-sm-6">
                    <div class="card shadow border-0 p-3 ms-3">
                        <embed id="view-pdf" src="" width="100%" height="600px" />
                    </div>
                </div>
                <div class="col-sm-6">
                    <div class="card shadow border-0 p-5 me-3">
                        <div id="loader" class="text-center">
                            <i class="fa-solid fa-spinner fa-spin-pulse font-large"></i>
                        </div>
                        <div id="download" class="text-center">
                            <a href="" id="download-btn" class="btn btn-md btn-warning" download><i class="fas fa-download font-large"></i></a>
                        </div>
                    </div>
                </div>
            </div>
        </div>
    </section>
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.1/dist/js/bootstrap.bundle.min.js" integrity="sha384-HwwvtgBNo3bZJJLYd8oVXjrBZt8cqVSpeBNS5n7C8IVInixGAoxmnlMuBnhbgrkm" crossorigin="anonymous"></script>
    <script src="https://kit.fontawesome.com/1da99de032.js" crossorigin="anonymous"></script>
    <script src="https://code.jquery.com/jquery-3.5.1.js"></script>
    <script src="//cdn.jsdelivr.net/npm/sweetalert2@11"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/pdf.js/2.11.338/pdf.min.js"></script>

    <script>
        let result = document.getElementById('result');
        let loader = document.getElementById('loader');
        let download = document.getElementById('download');
        let viewPdf = document.getElementById('view-pdf');
        let downloadBtn = document.getElementById('download-btn');

        $(document).ready(function () {
            $("#upload-btn").click(async function (event) {
                event.preventDefault();
                const formData = new FormData();
                const fileInput = document.getElementById('pdf-file') ;  
                var file = fileInput.files[0];           
                
                formData.append('pdf_file', file);
                formData.append('filename', file.name)
                let response = await fetch('/upload', {
                    method: "POST",
                    body: formData                
                });                
                processUploadResponse(response);  
            });
        });

        async function processUploadResponse(response){
            switch (response.status) {
                case 400:  
                    Swal.fire({
                        icon: 'error',
                        title: 'Oops!!!',
                        text: "Sorry, Couldn't be able to upload your pdf!!!",
                        confirmButtonColor: "#15011d"
                    }).then(function() {
                        window.location.reload();
                    });
                  break;
                case 200:                 
                    var json = await response.json();
                    if (json.msg == "error") {
                        Swal.fire({
                            icon: 'error',
                            title: 'Oops!',
                            text: 'Maximum number of pages exceeded.',
                            confirmButtonColor: "#545454"
                        }).then(function() {
                            window.location.reload();
                        });
                    }else {
                        result.style.display = "block";
                        loader.style.display = "block";
                        download.style.display = "none";
                        viewPdf.setAttribute('src', "../"+json.pdf_filename)
                        viewPdf.setAttribute('preload', 'auto');
                        const formData = new FormData();
                        formData.append('pdf_filename', json.pdf_filename)
                        fetch('/analyze', {
                            method: "POST",
                            body: formData                
                        }).then(processAnalyzeResponse)  
                    }
                    
                    break;
                default:
                    Swal.fire({
                        icon: 'error',
                        title: 'Oops!!!',
                        text: "There is a "+response.status+" error. Please contact admin for support.",
                        confirmButtonColor: "#15011d"
                    }).then(function() {
                        window.location.reload();
                    });
            }
        }

        async function processAnalyzeResponse(response){            
            switch (response.status) {
                case 400:  
                    Swal.fire({
                        icon: 'error',
                        title: 'Oops!!!',
                        text: "Sorry, Couldn't be able to analyze your pdf!!!",
                        confirmButtonColor: "#15011d"
                    }).then(function() {
                        window.location.reload();
                    });
                  break;
                case 200:                     
                    loader.style.display = "none";
                    download.style.display = "block";
                    var json = await response.json();
                    downloadBtn.setAttribute('href', "../"+json.output_file)
                    break;
                default:
                    Swal.fire({
                        icon: 'error',
                        title: 'Oops!!!',
                        text: "There is a "+response.status+" error. Please contact admin for support.",
                        confirmButtonColor: "#15011d"
                    }).then(function() {
                        window.location.reload();
                    });
            }
        }

        
    </script>
</body>
</html>

In [ ]:
# this is the app.py file

# =========================================================
# STEP 1: Define All Require Libraries
# =========================================================
from fastapi import FastAPI, Form, Request, Response, File, Depends, HTTPException, status
from fastapi.responses import RedirectResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
from fastapi.encoders import jsonable_encoder
import uvicorn
import os
import aiofiles
import json
import csv
from src.helper import llm_pipeline

# =========================================================
# STEP 2: Initialize the FastAPI
# =========================================================
app = FastAPI()

# store the loaded pdf into the static directory
app.mount("/static", StaticFiles(directory="static"), name="static")

templates = Jinja2Templates(directory="templates")

# =========================================================
# STEP 3: Initialize the Default Route
# =========================================================
@app.get("/")
async def index(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})


# =========================================================
# STEP 4 : To Upload the PDF
# =========================================================
@app.post("/upload")
async def chat(request: Request, pdf_file: bytes = File(), filename: str = Form(...)):
    # uploaded file store inside the static/docs/.pdf file
    base_folder = 'static/docs/'
    # first check and create folder 
    if not os.path.isdir(base_folder):
        os.mkdir(base_folder)
    # join the docs into uploaded pdf    
    pdf_filename = os.path.join(base_folder, filename)

    # save the pdf 
    async with aiofiles.open(pdf_filename, 'wb') as f:
        await f.write(pdf_file)

    # 
    response_data = jsonable_encoder(json.dumps({"msg": 'success',"pdf_filename": pdf_filename}))
    res = Response(response_data)
    return res

# =========================================================
# STEP 5 : Load the file and Create Question and Answer
# =========================================================
def get_csv(file_path):
    answer_generation_chain, ques_list = llm_pipeline(file_path)
    base_folder = 'static/output/'
    if not os.path.isdir(base_folder):
        os.mkdir(base_folder)
    output_file = base_folder+"QA.csv"
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(["Question", "Answer"])  # Writing the header row

        for question in ques_list:
            print("Question: ", question)
            answer = answer_generation_chain.run(question)
            print("Answer: ", answer)
            print("--------------------------------------------------\n\n")

            # Save answer to CSV file
            csv_writer.writerow([question, answer])
    return output_file

# =================================================================
# STEP: 6 -> Show the csv file (generated question) into frontend
# =================================================================
@app.post("/analyze")
async def chat(request: Request, pdf_filename: str = Form(...)):
    output_file = get_csv(pdf_filename)
    response_data = jsonable_encoder(json.dumps({"output_file": output_file}))
    res = Response(response_data)
    return res


# =========================================================
# STEP : To Run the App
# =========================================================
if __name__ == "__main__":
    uvicorn.run("app:app", host='0.0.0.0', port=8080, reload=True)
